# 🧠 Deep Learning Corporate Training Notebook
### A Comprehensive Guide to Neural Networks with PyTorch
---
**Audience:** Data scientists, ML engineers, and technical stakeholders  
**Hardware target:** NVIDIA GTX 1650 · 4 GB VRAM (CPU fallback included)  
**Framework:** PyTorch 2.x  
**Datasets:** Small built-in datasets — all training runs in minutes  

---
## Table of Contents
1. [Environment Setup](#1-environment-setup)
2. [Feedforward Neural Network (MLP)](#2-mlp)
3. [Convolutional Neural Network (CNN)](#3-cnn)
4. [Recurrent Neural Network (RNN)](#4-rnn)
5. [Long Short-Term Memory (LSTM)](#5-lstm)
6. [Gated Recurrent Unit (GRU)](#6-gru)
7. [Autoencoder (AE)](#7-autoencoder)
8. [Variational Autoencoder (VAE)](#8-vae)
9. [Generative Adversarial Network (GAN)](#9-gan)
10. [Transfer Learning](#10-transfer-learning)
11. [Key Concepts Cheat Sheet](#11-cheat-sheet)


---
## 1. Environment Setup <a id='1-environment-setup'></a>
Install dependencies, configure the device, and define shared utilities used throughout the notebook.


In [ ]:
# Install required packages (run once)
# !pip install torch torchvision matplotlib numpy scikit-learn tqdm


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset, Subset
import torchvision
import torchvision.transforms as transforms
import torchvision.models as models

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from sklearn.datasets import make_classification, make_regression
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

import warnings, time, math, random, os
warnings.filterwarnings('ignore')

# ── Reproducibility ──────────────────────────────────────────────────────────
SEED = 42
random.seed(SEED); np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# ── Device (GTX 1650 4 GB or CPU) ───────────────────────────────────────────
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"✅ Using device: {DEVICE}")
if DEVICE.type == 'cuda':
    props = torch.cuda.get_device_properties(0)
    print(f"   GPU : {props.name}")
    print(f"   VRAM: {props.total_memory / 1e9:.2f} GB")

# ── Helper: clear GPU memory between sections ────────────────────────────────
def free_gpu():
    if DEVICE.type == 'cuda':
        torch.cuda.empty_cache()
    import gc; gc.collect()

# ── Helper: pretty training bar ─────────────────────────────────────────────
def train_loop(model, loader, criterion, optimizer, epoch, tag='Train'):
    model.train()
    running_loss, correct, total = 0.0, 0, 0
    for X, y in loader:
        X, y = X.to(DEVICE), y.to(DEVICE)
        optimizer.zero_grad()
        out = model(X)
        loss = criterion(out, y)
        loss.backward()
        optimizer.step()
        running_loss += loss.item() * X.size(0)
        if out.shape[-1] > 1:          # classification
            pred = out.argmax(1)
            correct += (pred == y).sum().item()
            total += y.size(0)
    avg_loss = running_loss / len(loader.dataset)
    acc = correct / total * 100 if total else 0
    print(f"  [{tag}] Epoch {epoch:02d} | loss={avg_loss:.4f} | acc={acc:.1f}%")
    return avg_loss, acc

@torch.no_grad()
def eval_loop(model, loader, criterion, tag='Val'):
    model.eval()
    running_loss, correct, total = 0.0, 0, 0
    for X, y in loader:
        X, y = X.to(DEVICE), y.to(DEVICE)
        out = model(X)
        loss = criterion(out, y)
        running_loss += loss.item() * X.size(0)
        if out.shape[-1] > 1:
            pred = out.argmax(1)
            correct += (pred == y).sum().item()
            total += y.size(0)
    avg_loss = running_loss / len(loader.dataset)
    acc = correct / total * 100 if total else 0
    print(f"  [{tag}]         | loss={avg_loss:.4f} | acc={acc:.1f}%")
    return avg_loss, acc

def plot_history(train_losses, val_losses, title='Training History'):
    fig, ax = plt.subplots(figsize=(7, 3))
    ax.plot(train_losses, label='Train loss')
    ax.plot(val_losses,   label='Val loss')
    ax.set_xlabel('Epoch'); ax.set_ylabel('Loss')
    ax.set_title(title); ax.legend(); ax.grid(True, alpha=0.3)
    plt.tight_layout(); plt.show()

print("\n🛠️  All imports and helpers loaded.")


### Shared Dataset — MNIST (handwritten digits)
MNIST is a 28×28 greyscale image dataset with 10 classes (digits 0–9).  
We keep **10 000 train / 2 000 val samples** so every model trains in under 2 minutes on a GTX 1650.


In [ ]:
BATCH = 64
N_TRAIN, N_VAL = 10_000, 2_000   # small subsets — plenty for learning

transform_flat  = transforms.Compose([transforms.ToTensor(),
                                      transforms.Normalize((0.1307,), (0.3081,)),
                                      transforms.Lambda(lambda x: x.view(-1))])  # for MLP

transform_2d    = transforms.Compose([transforms.ToTensor(),
                                      transforms.Normalize((0.1307,), (0.3081,))])  # for CNN

def get_mnist(transform, n_train=N_TRAIN, n_val=N_VAL, batch=BATCH):
    full_train = torchvision.datasets.MNIST(root='./data', train=True,
                                            download=True, transform=transform)
    full_val   = torchvision.datasets.MNIST(root='./data', train=False,
                                            download=True, transform=transform)
    tr = DataLoader(Subset(full_train, range(n_train)), batch_size=batch, shuffle=True)
    va = DataLoader(Subset(full_val,   range(n_val)),   batch_size=batch, shuffle=False)
    return tr, va

print("📦 MNIST loader ready. Data will be downloaded on first use (~11 MB).")


---
## 2. Feedforward Neural Network (MLP) <a id='2-mlp'></a>

### What is an MLP?
A **Multi-Layer Perceptron** is the foundational building block of deep learning.  
Each layer applies a linear transformation followed by a non-linear activation:

```
output = activation(W · input + b)
```

**Architecture used here:**
```
Input (784) → Dense(512) → ReLU → Dropout(0.3)
           → Dense(256) → ReLU → Dropout(0.3)
           → Dense(128) → ReLU
           → Dense(10)  → Softmax
```

**Key hyperparameters:**
| Parameter | Value | Notes |
|-----------|-------|-------|
| Optimizer | Adam | Adaptive learning rate |
| LR | 1e-3 | Standard starting point |
| Batch size | 64 | Fits GTX 1650 easily |
| Epochs | 10 | Converges quickly |
| Dropout | 0.3 | Regularisation |


In [ ]:
# ── 2a. Dataset ───────────────────────────────────────────────────────────────
train_loader_flat, val_loader_flat = get_mnist(transform_flat)
print(f"Train batches: {len(train_loader_flat)} | Val batches: {len(val_loader_flat)}")

# Show sample images
sample_imgs, sample_labels = next(iter(
    DataLoader(Subset(torchvision.datasets.MNIST('./data', train=True,
               download=True, transform=transforms.ToTensor()), range(8)),
               batch_size=8)))
fig, axes = plt.subplots(1, 8, figsize=(12, 2))
for i, ax in enumerate(axes):
    ax.imshow(sample_imgs[i].squeeze(), cmap='gray')
    ax.set_title(f'Label: {sample_labels[i].item()}', fontsize=8)
    ax.axis('off')
plt.suptitle('MNIST Sample Images', fontsize=10)
plt.tight_layout(); plt.show()


In [ ]:
# ── 2b. Model Definition ──────────────────────────────────────────────────────
class MLP(nn.Module):
    def __init__(self, input_dim=784, hidden=[512, 256, 128], num_classes=10, dropout=0.3):
        super().__init__()
        layers = []
        in_feat = input_dim
        for h in hidden:
            layers += [nn.Linear(in_feat, h), nn.ReLU(), nn.Dropout(dropout)]
            in_feat = h
        layers.append(nn.Linear(in_feat, num_classes))
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x)

mlp = MLP().to(DEVICE)
total_params = sum(p.numel() for p in mlp.parameters())
print(f"MLP architecture:\n{mlp}")
print(f"\nTotal parameters: {total_params:,}")


In [ ]:
# ── 2c. Training ──────────────────────────────────────────────────────────────
EPOCHS = 10
mlp_criterion = nn.CrossEntropyLoss()
mlp_optimizer = optim.Adam(mlp.parameters(), lr=1e-3)
mlp_scheduler = optim.lr_scheduler.StepLR(mlp_optimizer, step_size=4, gamma=0.5)

mlp_train_losses, mlp_val_losses = [], []

print("Training MLP on MNIST...")
t0 = time.time()
for epoch in range(1, EPOCHS + 1):
    tl, _ = train_loop(mlp, train_loader_flat, mlp_criterion, mlp_optimizer, epoch)
    vl, _ = eval_loop(mlp,  val_loader_flat,   mlp_criterion)
    mlp_train_losses.append(tl); mlp_val_losses.append(vl)
    mlp_scheduler.step()

print(f"\n⏱️  Done in {time.time()-t0:.1f}s")
plot_history(mlp_train_losses, mlp_val_losses, 'MLP — Loss History')


In [ ]:
# ── 2d. Evaluation & Visualisation ───────────────────────────────────────────
mlp.eval()
all_preds, all_labels = [], []
with torch.no_grad():
    for X, y in val_loader_flat:
        preds = mlp(X.to(DEVICE)).argmax(1).cpu()
        all_preds.extend(preds.numpy()); all_labels.extend(y.numpy())

from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, classification_report
cm = confusion_matrix(all_labels, all_preds)
fig, ax = plt.subplots(figsize=(7, 6))
ConfusionMatrixDisplay(cm, display_labels=list(range(10))).plot(ax=ax, colorbar=False)
ax.set_title('MLP Confusion Matrix (Validation)'); plt.tight_layout(); plt.show()

print("\nClassification Report:")
print(classification_report(all_labels, all_preds, target_names=[str(i) for i in range(10)]))
free_gpu()


---
## 3. Convolutional Neural Network (CNN) <a id='3-cnn'></a>

### What is a CNN?
CNNs exploit the **spatial structure** of images using learnable filters (kernels).  
A convolution slides a filter across the image to produce a feature map:

```
Feature map[i,j] = Σ(kernel * input_patch)
```

**Architecture (LeNet-inspired):**
```
Input (1×28×28)
  → Conv(1→32, 3×3) → BN → ReLU → MaxPool(2×2)     [→ 32×13×13]
  → Conv(32→64, 3×3)→ BN → ReLU → MaxPool(2×2)     [→ 64×5×5 ]
  → Flatten → Dense(512) → ReLU → Dropout → Dense(10)
```

**Key concepts:**
- **Kernel / Filter** — learnable weight matrix detecting local features  
- **Stride** — how many pixels the kernel moves at each step  
- **Padding** — zeros added around the image to control output size  
- **MaxPooling** — reduces spatial size, adds translation invariance  
- **Batch Normalisation** — stabilises activations, speeds training  


In [ ]:
# ── 3a. Dataset ───────────────────────────────────────────────────────────────
train_loader_2d, val_loader_2d = get_mnist(transform_2d)

# ── 3b. Model Definition ──────────────────────────────────────────────────────
class CNN(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 32, kernel_size=3, padding=1),   # 28×28 → 28×28
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(2),                               # → 14×14

            nn.Conv2d(32, 64, kernel_size=3, padding=1),  # 14×14 → 14×14
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2),                               # → 7×7
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(64 * 7 * 7, 512),
            nn.ReLU(),
            nn.Dropout(0.4),
            nn.Linear(512, num_classes),
        )

    def forward(self, x):
        return self.classifier(self.features(x))

cnn = CNN().to(DEVICE)
print(f"CNN architecture:\n{cnn}")
print(f"\nTotal parameters: {sum(p.numel() for p in cnn.parameters()):,}")


In [ ]:
# ── 3c. Training ──────────────────────────────────────────────────────────────
cnn_criterion = nn.CrossEntropyLoss()
cnn_optimizer = optim.Adam(cnn.parameters(), lr=1e-3, weight_decay=1e-4)
cnn_scheduler = optim.lr_scheduler.CosineAnnealingLR(cnn_optimizer, T_max=10)

cnn_train_losses, cnn_val_losses = [], []
print("Training CNN on MNIST...")
t0 = time.time()
for epoch in range(1, EPOCHS + 1):
    tl, _ = train_loop(cnn, train_loader_2d, cnn_criterion, cnn_optimizer, epoch)
    vl, _ = eval_loop(cnn,  val_loader_2d,   cnn_criterion)
    cnn_train_losses.append(tl); cnn_val_losses.append(vl)
    cnn_scheduler.step()

print(f"\n⏱️  Done in {time.time()-t0:.1f}s")
plot_history(cnn_train_losses, cnn_val_losses, 'CNN — Loss History')


In [ ]:
# ── 3d. Visualise feature maps ────────────────────────────────────────────────
cnn.eval()
sample_img, _ = next(iter(val_loader_2d))
sample_img = sample_img[:1].to(DEVICE)

with torch.no_grad():
    feat_maps = cnn.features[:3](sample_img)   # after first Conv+BN+ReLU

feat_maps = feat_maps.squeeze().cpu().numpy()  # (32, 28, 28)
fig, axes = plt.subplots(4, 8, figsize=(14, 7))
for i, ax in enumerate(axes.flat):
    ax.imshow(feat_maps[i], cmap='viridis')
    ax.set_title(f'F{i}', fontsize=7); ax.axis('off')
plt.suptitle('First-layer CNN Feature Maps', fontsize=12)
plt.tight_layout(); plt.show()
free_gpu()


---
## 4. Recurrent Neural Network (RNN) <a id='4-rnn'></a>

### What is an RNN?
RNNs process **sequences** by maintaining a hidden state that carries information across timesteps:

```
h_t = tanh(W_hh · h_{t-1} + W_xh · x_t + b)
```

**Use case:** Sentiment classification on a synthetic word-index dataset.  
We generate sentences of length 20 from a vocabulary of 100 words, labelled binary (positive/negative).

**Limitation of vanilla RNNs:**  
Gradients vanish over long sequences — which is why LSTMs and GRUs were invented.


In [ ]:
# ── 4a. Synthetic Sequence Dataset ───────────────────────────────────────────
VOCAB_SIZE   = 100
SEQ_LEN      = 20
N_SAMPLES    = 4000
EMBED_DIM    = 16

# Positive sequences have higher word indices on average
np.random.seed(SEED)
pos = np.random.randint(50, VOCAB_SIZE, (N_SAMPLES//2, SEQ_LEN))
neg = np.random.randint(0,  50,         (N_SAMPLES//2, SEQ_LEN))
X_seq = np.vstack([pos, neg])
y_seq = np.array([1]*(N_SAMPLES//2) + [0]*(N_SAMPLES//2))

idx = np.random.permutation(N_SAMPLES)
X_seq, y_seq = X_seq[idx], y_seq[idx]

split = int(0.8 * N_SAMPLES)
X_tr, X_va = X_seq[:split], X_seq[split:]
y_tr, y_va = y_seq[:split], y_seq[split:]

def seq_loader(X, y, batch=BATCH):
    ds = TensorDataset(torch.LongTensor(X), torch.LongTensor(y))
    return DataLoader(ds, batch_size=batch, shuffle=True)

rnn_train = seq_loader(X_tr, y_tr)
rnn_val   = seq_loader(X_va, y_va)
print(f"Sequence dataset — train: {X_tr.shape}, val: {X_va.shape}")


In [ ]:
# ── 4b. Model Definition ──────────────────────────────────────────────────────
class VanillaRNN(nn.Module):
    def __init__(self, vocab_size=VOCAB_SIZE, embed_dim=EMBED_DIM,
                 hidden_size=64, num_layers=1, num_classes=2):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim)
        self.rnn = nn.RNN(embed_dim, hidden_size, num_layers,
                          batch_first=True, nonlinearity='tanh')
        self.fc  = nn.Linear(hidden_size, num_classes)

    def forward(self, x):
        emb = self.embedding(x)           # (B, T, E)
        out, _ = self.rnn(emb)            # (B, T, H)
        last  = out[:, -1, :]             # take last timestep
        return self.fc(last)

rnn_model = VanillaRNN().to(DEVICE)
print(rnn_model)
print(f"Parameters: {sum(p.numel() for p in rnn_model.parameters()):,}")


In [ ]:
# ── 4c. Training ──────────────────────────────────────────────────────────────
rnn_criterion = nn.CrossEntropyLoss()
rnn_optimizer = optim.Adam(rnn_model.parameters(), lr=5e-3)

rnn_train_losses, rnn_val_losses = [], []
print("Training Vanilla RNN...")
t0 = time.time()
for epoch in range(1, 11):
    tl, _ = train_loop(rnn_model, rnn_train, rnn_criterion, rnn_optimizer, epoch)
    vl, _ = eval_loop(rnn_model,  rnn_val,   rnn_criterion)
    rnn_train_losses.append(tl); rnn_val_losses.append(vl)

print(f"\n⏱️  Done in {time.time()-t0:.1f}s")
plot_history(rnn_train_losses, rnn_val_losses, 'Vanilla RNN — Loss History')
free_gpu()


---
## 5. Long Short-Term Memory (LSTM) <a id='5-lstm'></a>

### What is an LSTM?
LSTMs solve the vanishing gradient problem through **gating mechanisms**:

| Gate | Symbol | Purpose |
|------|--------|---------|
| Forget gate | f_t | Decide what to discard from cell state |
| Input gate  | i_t | Decide what new information to store |
| Cell update | g_t | Candidate new values |
| Output gate | o_t | Decide what to output |

```
f_t = σ(W_f · [h_{t-1}, x_t] + b_f)
i_t = σ(W_i · [h_{t-1}, x_t] + b_i)
C_t = f_t ⊙ C_{t-1} + i_t ⊙ tanh(W_c · [h_{t-1}, x_t] + b_c)
h_t = o_t ⊙ tanh(C_t)
```

**Use case:** Time-series sine wave regression — predict next value from a window of 30 past values.


In [ ]:
# ── 5a. Sine-wave Dataset ─────────────────────────────────────────────────────
WINDOW = 30

t_vals = np.linspace(0, 100 * np.pi, 6000)
signal = np.sin(t_vals) + 0.1 * np.random.randn(len(t_vals))   # noisy sine

def make_windows(sig, win):
    X = np.array([sig[i:i+win]     for i in range(len(sig)-win-1)], dtype=np.float32)
    y = np.array([sig[i+win]       for i in range(len(sig)-win-1)], dtype=np.float32)
    return X[:, :, None], y[:, None]   # (N, T, 1), (N, 1)

X_ts, y_ts = make_windows(signal, WINDOW)
split = int(0.8 * len(X_ts))
lstm_train = DataLoader(TensorDataset(torch.tensor(X_ts[:split]),
                                       torch.tensor(y_ts[:split])),
                         batch_size=128, shuffle=True)
lstm_val   = DataLoader(TensorDataset(torch.tensor(X_ts[split:]),
                                       torch.tensor(y_ts[split:])),
                         batch_size=128, shuffle=False)

plt.figure(figsize=(10, 2))
plt.plot(signal[:300], color='steelblue')
plt.title('Noisy Sine Wave (first 300 points)'); plt.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()
print(f"Train windows: {X_ts[:split].shape}  | Val windows: {X_ts[split:].shape}")


In [ ]:
# ── 5b. LSTM Model ────────────────────────────────────────────────────────────
class LSTMRegressor(nn.Module):
    def __init__(self, input_size=1, hidden=64, num_layers=2, dropout=0.2):
        super().__init__()
        self.lstm = nn.LSTM(input_size, hidden, num_layers,
                            batch_first=True, dropout=dropout)
        self.fc   = nn.Linear(hidden, 1)

    def forward(self, x):
        out, _ = self.lstm(x)
        return self.fc(out[:, -1, :])

lstm_model = LSTMRegressor().to(DEVICE)
print(lstm_model)
print(f"Parameters: {sum(p.numel() for p in lstm_model.parameters()):,}")


In [ ]:
# ── 5c. Training ──────────────────────────────────────────────────────────────
lstm_criterion = nn.MSELoss()
lstm_optimizer = optim.Adam(lstm_model.parameters(), lr=1e-3)
lstm_scheduler = optim.lr_scheduler.ReduceLROnPlateau(lstm_optimizer, patience=3, factor=0.5)

lstm_train_losses, lstm_val_losses = [], []

def reg_train_loop(model, loader, criterion, optimizer, epoch):
    model.train()
    total_loss = 0
    for X, y in loader:
        X, y = X.to(DEVICE), y.to(DEVICE)
        optimizer.zero_grad()
        pred = model(X)
        loss = criterion(pred, y)
        loss.backward(); optimizer.step()
        total_loss += loss.item() * X.size(0)
    avg = total_loss / len(loader.dataset)
    print(f"  [Train] Epoch {epoch:02d} | MSE={avg:.5f}")
    return avg

@torch.no_grad()
def reg_eval_loop(model, loader, criterion):
    model.eval()
    total_loss = 0
    for X, y in loader:
        X, y = X.to(DEVICE), y.to(DEVICE)
        pred = model(X)
        total_loss += criterion(pred, y).item() * X.size(0)
    avg = total_loss / len(loader.dataset)
    print(f"  [Val  ]         | MSE={avg:.5f}")
    return avg

print("Training LSTM on sine-wave regression...")
t0 = time.time()
for epoch in range(1, 16):
    tl = reg_train_loop(lstm_model, lstm_train, lstm_criterion, lstm_optimizer, epoch)
    vl = reg_eval_loop(lstm_model,  lstm_val,   lstm_criterion)
    lstm_train_losses.append(tl); lstm_val_losses.append(vl)
    lstm_scheduler.step(vl)

print(f"\n⏱️  Done in {time.time()-t0:.1f}s")
plot_history(lstm_train_losses, lstm_val_losses, 'LSTM — MSE Loss History')


In [ ]:
# ── 5d. Prediction vs Ground Truth ───────────────────────────────────────────
lstm_model.eval()
X_sample = torch.tensor(X_ts[split:split+200]).to(DEVICE)
with torch.no_grad():
    preds = lstm_model(X_sample).cpu().numpy().flatten()
gt = y_ts[split:split+200].flatten()

plt.figure(figsize=(10, 3))
plt.plot(gt,    label='Ground Truth', alpha=0.8)
plt.plot(preds, label='LSTM Prediction', alpha=0.8, linestyle='--')
plt.title('LSTM Sine-Wave Prediction (first 200 val samples)')
plt.legend(); plt.grid(True, alpha=0.3); plt.tight_layout(); plt.show()
free_gpu()


---
## 6. Gated Recurrent Unit (GRU) <a id='6-gru'></a>

### GRU vs LSTM
GRU is a **simplified LSTM** with only two gates, making it faster to train with comparable performance:

| | LSTM | GRU |
|-|------|-----|
| Gates | 3 (forget, input, output) | 2 (reset, update) |
| Cell state | Separate C_t | Merged into h_t |
| Parameters | More | ~25% fewer |
| Speed | Slower | Faster |

**When to use GRU:** Smaller datasets, tighter latency budgets, or when LSTM is overfitting.


In [ ]:
# ── 6a. GRU Model (reuse sine-wave data from section 5) ─────────────────────
class GRURegressor(nn.Module):
    def __init__(self, input_size=1, hidden=64, num_layers=2, dropout=0.2):
        super().__init__()
        self.gru = nn.GRU(input_size, hidden, num_layers,
                          batch_first=True, dropout=dropout)
        self.fc  = nn.Linear(hidden, 1)

    def forward(self, x):
        out, _ = self.gru(x)
        return self.fc(out[:, -1, :])

gru_model     = GRURegressor().to(DEVICE)
gru_criterion = nn.MSELoss()
gru_optimizer = optim.Adam(gru_model.parameters(), lr=1e-3)
gru_train_losses, gru_val_losses = [], []

print("Training GRU on sine-wave regression...")
t0 = time.time()
for epoch in range(1, 16):
    tl = reg_train_loop(gru_model, lstm_train, gru_criterion, gru_optimizer, epoch)
    vl = reg_eval_loop(gru_model,  lstm_val,   gru_criterion)
    gru_train_losses.append(tl); gru_val_losses.append(vl)

print(f"\n⏱️  Done in {time.time()-t0:.1f}s")

# ── 6b. Compare LSTM vs GRU ──────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(8, 3))
ax.plot(lstm_val_losses, label='LSTM val MSE', marker='o', markersize=3)
ax.plot(gru_val_losses,  label='GRU  val MSE', marker='s', markersize=3)
ax.set_xlabel('Epoch'); ax.set_ylabel('MSE'); ax.set_title('LSTM vs GRU — Validation Loss')
ax.legend(); ax.grid(True, alpha=0.3); plt.tight_layout(); plt.show()

lstm_params = sum(p.numel() for p in lstm_model.parameters())
gru_params  = sum(p.numel() for p in gru_model.parameters())
print(f"LSTM params: {lstm_params:,}   GRU params: {gru_params:,}  "
      f"(GRU is {(1-gru_params/lstm_params)*100:.1f}% smaller)")
free_gpu()


---
## 7. Autoencoder (AE) <a id='7-autoencoder'></a>

### What is an Autoencoder?
An Autoencoder learns a **compressed representation (latent code)** of the input by training
the network to reconstruct the input from the bottleneck:

```
Input → Encoder → Latent (z) → Decoder → Reconstructed Input
```

**Loss:** Reconstruction loss (MSE or Binary Cross-Entropy)

**Applications:**
- Dimensionality reduction (alternative to PCA)
- Anomaly detection (high reconstruction error = anomaly)
- Data denoising
- Feature extraction


In [ ]:
# ── 7a. Dataset ───────────────────────────────────────────────────────────────
ae_transform = transforms.Compose([transforms.ToTensor()])
ae_train, ae_val = get_mnist(ae_transform)

# ── 7b. Model Definition ──────────────────────────────────────────────────────
class Autoencoder(nn.Module):
    def __init__(self, latent_dim=32):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Flatten(),
            nn.Linear(784, 256), nn.ReLU(),
            nn.Linear(256, 128), nn.ReLU(),
            nn.Linear(128, latent_dim),
        )
        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, 128), nn.ReLU(),
            nn.Linear(128, 256),        nn.ReLU(),
            nn.Linear(256, 784),        nn.Sigmoid(),   # pixel values in [0,1]
        )

    def forward(self, x):
        z    = self.encoder(x)
        recon = self.decoder(z)
        return recon.view(-1, 1, 28, 28), z

ae_model = Autoencoder(latent_dim=32).to(DEVICE)
print(ae_model)
print(f"Parameters: {sum(p.numel() for p in ae_model.parameters()):,}")


In [ ]:
# ── 7c. Training ──────────────────────────────────────────────────────────────
ae_criterion = nn.MSELoss()
ae_optimizer = optim.Adam(ae_model.parameters(), lr=1e-3)
ae_train_losses, ae_val_losses = [], []

def ae_train_loop(model, loader, criterion, optimizer, epoch):
    model.train()
    total_loss = 0
    for X, _ in loader:
        X = X.to(DEVICE)
        optimizer.zero_grad()
        recon, _ = model(X)
        loss = criterion(recon, X)
        loss.backward(); optimizer.step()
        total_loss += loss.item() * X.size(0)
    avg = total_loss / len(loader.dataset)
    print(f"  [Train] Epoch {epoch:02d} | MSE={avg:.5f}")
    return avg

@torch.no_grad()
def ae_eval_loop(model, loader, criterion):
    model.eval()
    total_loss = 0
    for X, _ in loader:
        X = X.to(DEVICE)
        recon, _ = model(X)
        total_loss += criterion(recon, X).item() * X.size(0)
    avg = total_loss / len(loader.dataset)
    print(f"  [Val  ]         | MSE={avg:.5f}")
    return avg

print("Training Autoencoder on MNIST...")
t0 = time.time()
for epoch in range(1, 11):
    tl = ae_train_loop(ae_model, ae_train, ae_criterion, ae_optimizer, epoch)
    vl = ae_eval_loop(ae_model,  ae_val,   ae_criterion)
    ae_train_losses.append(tl); ae_val_losses.append(vl)

print(f"\n⏱️  Done in {time.time()-t0:.1f}s")
plot_history(ae_train_losses, ae_val_losses, 'Autoencoder — Reconstruction Loss')


In [ ]:
# ── 7d. Visualise Reconstructions ────────────────────────────────────────────
ae_model.eval()
X_sample, _ = next(iter(ae_val))
X_sample = X_sample[:8].to(DEVICE)
with torch.no_grad():
    recon, _ = ae_model(X_sample)

fig, axes = plt.subplots(2, 8, figsize=(14, 4))
for i in range(8):
    axes[0, i].imshow(X_sample[i].cpu().squeeze(), cmap='gray')
    axes[0, i].set_title('Original', fontsize=7); axes[0, i].axis('off')
    axes[1, i].imshow(recon[i].cpu().squeeze(),    cmap='gray')
    axes[1, i].set_title('Recon',    fontsize=7); axes[1, i].axis('off')
plt.suptitle('Autoencoder: Original vs Reconstructed', fontsize=11)
plt.tight_layout(); plt.show()

# ── 7e. 2D Latent Space Visualisation (reduce to 2D via another AE) ──────────
from sklearn.decomposition import PCA
ae_model.eval()
all_z, all_labels = [], []
with torch.no_grad():
    for X, y in ae_val:
        _, z = ae_model(X.to(DEVICE))
        all_z.append(z.cpu()); all_labels.extend(y.numpy())

all_z = torch.cat(all_z).numpy()
z_2d  = PCA(n_components=2).fit_transform(all_z)

plt.figure(figsize=(7, 5))
sc = plt.scatter(z_2d[:, 0], z_2d[:, 1], c=all_labels, cmap='tab10', s=6, alpha=0.7)
plt.colorbar(sc, label='Digit class')
plt.title('Latent Space (PCA of 32-D codes, coloured by digit)')
plt.grid(True, alpha=0.3); plt.tight_layout(); plt.show()
free_gpu()


---
## 8. Variational Autoencoder (VAE) <a id='8-vae'></a>

### What is a VAE?
A VAE learns a **probabilistic** latent space. Instead of mapping to a single point,  
the encoder outputs a **mean (μ) and log-variance (log σ²)**:

```
z ~ N(μ, σ²)   (reparameterisation trick: z = μ + σ·ε,  ε ~ N(0,1))
```

**Loss = Reconstruction Loss + KL Divergence**  
The KL term regularises the latent space to be approximately Gaussian,  
enabling smooth **generation** of new samples by sampling z ~ N(0, I).


In [ ]:
# ── 8a. VAE Model ─────────────────────────────────────────────────────────────
LATENT = 16

class VAE(nn.Module):
    def __init__(self, latent_dim=LATENT):
        super().__init__()
        # Encoder
        self.enc_fc1 = nn.Linear(784, 256)
        self.enc_mu  = nn.Linear(256, latent_dim)
        self.enc_lv  = nn.Linear(256, latent_dim)   # log variance
        # Decoder
        self.dec = nn.Sequential(
            nn.Linear(latent_dim, 256), nn.ReLU(),
            nn.Linear(256, 784),        nn.Sigmoid(),
        )

    def encode(self, x):
        h  = F.relu(self.enc_fc1(x.view(-1, 784)))
        return self.enc_mu(h), self.enc_lv(h)

    def reparameterise(self, mu, logvar):
        if self.training:
            std = torch.exp(0.5 * logvar)
            eps = torch.randn_like(std)
            return mu + eps * std
        return mu

    def decode(self, z):
        return self.dec(z).view(-1, 1, 28, 28)

    def forward(self, x):
        mu, logvar = self.encode(x)
        z = self.reparameterise(mu, logvar)
        return self.decode(z), mu, logvar

def vae_loss(recon, x, mu, logvar, beta=1.0):
    bce  = F.binary_cross_entropy(recon, x, reduction='sum')
    kld  = -0.5 * torch.sum(1 + logvar - mu.pow(2) - logvar.exp())
    return (bce + beta * kld) / x.size(0)

vae = VAE().to(DEVICE)
print(vae)
print(f"Parameters: {sum(p.numel() for p in vae.parameters()):,}")


In [ ]:
# ── 8b. Training ──────────────────────────────────────────────────────────────
vae_optimizer = optim.Adam(vae.parameters(), lr=1e-3)
vae_train_losses, vae_val_losses = [], []

def vae_train_epoch(model, loader, optimizer, epoch, beta=1.0):
    model.train(); total = 0
    for X, _ in loader:
        X = X.to(DEVICE)
        optimizer.zero_grad()
        recon, mu, logvar = model(X)
        loss = vae_loss(recon, X, mu, logvar, beta)
        loss.backward(); optimizer.step()
        total += loss.item() * X.size(0)
    avg = total / len(loader.dataset)
    print(f"  [Train] Epoch {epoch:02d} | ELBO={avg:.2f}")
    return avg

@torch.no_grad()
def vae_eval_epoch(model, loader, beta=1.0):
    model.eval(); total = 0
    for X, _ in loader:
        X = X.to(DEVICE)
        recon, mu, logvar = model(X)
        total += vae_loss(recon, X, mu, logvar, beta).item() * X.size(0)
    avg = total / len(loader.dataset)
    print(f"  [Val  ]         | ELBO={avg:.2f}")
    return avg

print("Training VAE on MNIST...")
t0 = time.time()
for epoch in range(1, 11):
    tl = vae_train_epoch(vae, ae_train, vae_optimizer, epoch)
    vl = vae_eval_epoch(vae, ae_val)
    vae_train_losses.append(tl); vae_val_losses.append(vl)

print(f"\n⏱️  Done in {time.time()-t0:.1f}s")
plot_history(vae_train_losses, vae_val_losses, 'VAE — ELBO Loss')


In [ ]:
# ── 8c. Generate New Digits ───────────────────────────────────────────────────
vae.eval()
with torch.no_grad():
    z_sample = torch.randn(16, LATENT).to(DEVICE)
    gen_imgs  = vae.decode(z_sample).cpu()

fig, axes = plt.subplots(2, 8, figsize=(14, 4))
for i, ax in enumerate(axes.flat):
    ax.imshow(gen_imgs[i].squeeze(), cmap='gray'); ax.axis('off')
plt.suptitle('VAE — Randomly Generated Digits (z ~ N(0,I))', fontsize=11)
plt.tight_layout(); plt.show()

# ── 8d. Latent Interpolation ─────────────────────────────────────────────────
vae.eval()
X_pair, y_pair = next(iter(ae_val))
X_pair = X_pair.to(DEVICE)
with torch.no_grad():
    mu1, _  = vae.encode(X_pair[:1])
    mu2, _  = vae.encode(X_pair[1:2])
    alphas  = torch.linspace(0, 1, 10).to(DEVICE)
    interps = torch.stack([vae.decode(alpha*mu2 + (1-alpha)*mu1) for alpha in alphas]).squeeze()

fig, axes = plt.subplots(1, 10, figsize=(14, 2))
for i, ax in enumerate(axes):
    ax.imshow(interps[i].cpu().squeeze(), cmap='gray'); ax.axis('off')
plt.suptitle('VAE Latent Space Interpolation (left → right)', fontsize=11)
plt.tight_layout(); plt.show()
free_gpu()


---
## 9. Generative Adversarial Network (GAN) <a id='9-gan'></a>

### What is a GAN?
A GAN is a **two-player minimax game** between:

- **Generator (G):** Learns to produce realistic fake images from random noise z ~ N(0,I)
- **Discriminator (D):** Learns to distinguish real images from fakes

```
min_G  max_D  E[log D(x)] + E[log(1 - D(G(z)))]
```

**Training loop:**
1. Sample real batch → train D to output 1
2. Sample noise → generate fakes → train D to output 0
3. Sample new noise → generate fakes → train G to fool D (D outputs 1 for fakes)

> ⚡ **GTX 1650 note:** We use a **lightweight DCGAN** (small filters, 2 conv layers).  
> Batch size = 64, image = 28×28 — peak VRAM < 800 MB.


In [ ]:
# ── 9a. Dataset (same MNIST 2D) ───────────────────────────────────────────────
# transform: images in [-1, 1] for Tanh generator output
gan_transform = transforms.Compose([transforms.ToTensor(),
                                     transforms.Normalize((0.5,), (0.5,))])
gan_train_loader, _ = get_mnist(gan_transform, n_train=10_000, batch=64)

NOISE_DIM  = 64   # latent noise dimension
GAN_EPOCHS = 15

# ── 9b. Generator ─────────────────────────────────────────────────────────────
class Generator(nn.Module):
    """Noise (64,) → Conv-transpose layers → (1, 28, 28)"""
    def __init__(self, noise_dim=NOISE_DIM):
        super().__init__()
        self.proj = nn.Sequential(
            nn.Linear(noise_dim, 7*7*128),
            nn.ReLU(),
        )
        self.conv = nn.Sequential(
            nn.ConvTranspose2d(128, 64, 4, stride=2, padding=1),  # 7→14
            nn.BatchNorm2d(64), nn.ReLU(),
            nn.ConvTranspose2d(64,  1,  4, stride=2, padding=1),  # 14→28
            nn.Tanh(),
        )

    def forward(self, z):
        x = self.proj(z).view(-1, 128, 7, 7)
        return self.conv(x)

# ── 9c. Discriminator ─────────────────────────────────────────────────────────
class Discriminator(nn.Module):
    """(1, 28, 28) → real/fake logit"""
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(1, 64, 4, stride=2, padding=1),  # 28→14
            nn.LeakyReLU(0.2),
            nn.Conv2d(64, 128, 4, stride=2, padding=1), # 14→7
            nn.BatchNorm2d(128), nn.LeakyReLU(0.2),
            nn.Flatten(),
            nn.Linear(128*7*7, 1),
        )

    def forward(self, x):
        return self.net(x)

G = Generator().to(DEVICE)
D = Discriminator().to(DEVICE)
print(f"Generator params:     {sum(p.numel() for p in G.parameters()):,}")
print(f"Discriminator params: {sum(p.numel() for p in D.parameters()):,}")


In [ ]:
# ── 9d. Training ──────────────────────────────────────────────────────────────
g_opt = optim.Adam(G.parameters(), lr=2e-4, betas=(0.5, 0.999))
d_opt = optim.Adam(D.parameters(), lr=2e-4, betas=(0.5, 0.999))
gan_criterion = nn.BCEWithLogitsLoss()

fixed_noise = torch.randn(16, NOISE_DIM).to(DEVICE)   # for monitoring
d_losses, g_losses = [], []

print(f"Training GAN for {GAN_EPOCHS} epochs...")
t0 = time.time()

for epoch in range(1, GAN_EPOCHS + 1):
    d_epoch, g_epoch, n_batches = 0, 0, 0
    for real, _ in gan_train_loader:
        bs = real.size(0)
        real = real.to(DEVICE)

        # ── Train Discriminator ───────────────────────────────────────────
        z    = torch.randn(bs, NOISE_DIM).to(DEVICE)
        fake = G(z).detach()

        real_logits = D(real)
        fake_logits = D(fake)
        d_loss = (gan_criterion(real_logits, torch.ones_like(real_logits))
                + gan_criterion(fake_logits, torch.zeros_like(fake_logits))) / 2
        d_opt.zero_grad(); d_loss.backward(); d_opt.step()

        # ── Train Generator ───────────────────────────────────────────────
        z    = torch.randn(bs, NOISE_DIM).to(DEVICE)
        fake = G(z)
        g_logits = D(fake)
        g_loss = gan_criterion(g_logits, torch.ones_like(g_logits))
        g_opt.zero_grad(); g_loss.backward(); g_opt.step()

        d_epoch += d_loss.item(); g_epoch += g_loss.item(); n_batches += 1

    d_losses.append(d_epoch / n_batches)
    g_losses.append(g_epoch / n_batches)
    print(f"  Epoch {epoch:02d} | D_loss={d_losses[-1]:.4f} | G_loss={g_losses[-1]:.4f}")

print(f"\n⏱️  Done in {time.time()-t0:.1f}s")

fig, ax = plt.subplots(figsize=(7, 3))
ax.plot(d_losses, label='Discriminator'); ax.plot(g_losses, label='Generator')
ax.set_title('GAN Training Losses'); ax.set_xlabel('Epoch')
ax.legend(); ax.grid(True, alpha=0.3); plt.tight_layout(); plt.show()


In [ ]:
# ── 9e. Visualise Generated Samples ──────────────────────────────────────────
G.eval()
with torch.no_grad():
    gen_imgs = G(fixed_noise).cpu()

fig, axes = plt.subplots(2, 8, figsize=(14, 4))
for i, ax in enumerate(axes.flat):
    ax.imshow(gen_imgs[i].squeeze() * 0.5 + 0.5, cmap='gray')   # de-normalise
    ax.axis('off')
plt.suptitle('GAN Generated Digits (after training)', fontsize=11)
plt.tight_layout(); plt.show()
free_gpu()


---
## 10. Transfer Learning <a id='10-transfer-learning'></a>

### What is Transfer Learning?
Transfer learning reuses a **pre-trained model's feature extractor** and only trains  
a new classification head on the target task — drastically reducing data and compute needs.

```
Pre-trained backbone (frozen weights)  →  New head (trainable)
ImageNet features                          Your classes
```

**Strategy A — Feature extraction:** Freeze all backbone weights, train only the head.  
**Strategy B — Fine-tuning:** Unfreeze later layers, use a very low learning rate.

**Model used:** `MobileNetV2` (3.4 M parameters, ~14 MB) — ideal for GTX 1650.  
**Dataset:** CIFAR-10 subset (5 classes, 2 500 train / 500 val images, 32×32 RGB).


In [ ]:
# ── 10a. CIFAR-10 Subset (5 classes, small) ──────────────────────────────────
CLASSES   = [0, 1, 2, 3, 4]   # airplane, automobile, bird, cat, deer
CLASS_MAP = {c: i for i, c in enumerate(CLASSES)}
CLASS_NAMES = ['airplane', 'automobile', 'bird', 'cat', 'deer']
N_PER_CLASS = 500

tl_transform = transforms.Compose([
    transforms.Resize(64),
    transforms.CenterCrop(64),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

def subset_cifar(split='train', n_per_class=N_PER_CLASS):
    full = torchvision.datasets.CIFAR10('./data', train=(split=='train'),
                                         download=True, transform=tl_transform)
    indices = []
    counts  = {c: 0 for c in CLASSES}
    for i, (_, label) in enumerate(full):
        if label in CLASSES and counts[label] < n_per_class:
            indices.append(i); counts[label] += 1
        if all(v >= n_per_class for v in counts.values()):
            break
    return Subset(full, indices)

tl_train_ds = subset_cifar('train', 500)
tl_val_ds   = subset_cifar('val',   100)
tl_train = DataLoader(tl_train_ds, batch_size=32, shuffle=True)
tl_val   = DataLoader(tl_val_ds,   batch_size=32, shuffle=False)
print(f"Transfer train: {len(tl_train_ds)}  | Val: {len(tl_val_ds)}")


In [ ]:
# ── 10b. Load MobileNetV2, replace classifier head ───────────────────────────
backbone = models.mobilenet_v2(weights=models.MobileNet_V2_Weights.DEFAULT)

# Freeze all backbone layers
for param in backbone.parameters():
    param.requires_grad = False

# Replace the last classification layer
n_features = backbone.classifier[1].in_features
backbone.classifier[1] = nn.Linear(n_features, len(CLASSES))

tl_model = backbone.to(DEVICE)
trainable = sum(p.numel() for p in tl_model.parameters() if p.requires_grad)
total     = sum(p.numel() for p in tl_model.parameters())
print(f"Trainable params: {trainable:,} / {total:,}  ({trainable/total*100:.2f}%)")


In [ ]:
# ── 10c. Training — Feature Extraction Phase ─────────────────────────────────
tl_criterion = nn.CrossEntropyLoss()
tl_optimizer = optim.Adam(tl_model.classifier[1].parameters(), lr=1e-3)

tl_train_losses, tl_val_losses = [], []
print("Phase 1: Feature Extraction (frozen backbone, 8 epochs)...")
t0 = time.time()
for epoch in range(1, 9):
    tl, _ = train_loop(tl_model, tl_train, tl_criterion, tl_optimizer, epoch)
    vl, _ = eval_loop(tl_model,  tl_val,   tl_criterion)
    tl_train_losses.append(tl); tl_val_losses.append(vl)
print(f"⏱️  Phase 1 done in {time.time()-t0:.1f}s")


In [ ]:
# ── 10d. Fine-tuning Phase — Unfreeze last 3 blocks ─────────────────────────
for name, param in tl_model.named_parameters():
    if 'features.17' in name or 'features.18' in name or 'classifier' in name:
        param.requires_grad = True

ft_optimizer = optim.Adam(
    filter(lambda p: p.requires_grad, tl_model.parameters()),
    lr=1e-4, weight_decay=1e-4)   # very low LR for fine-tuning
ft_scheduler = optim.lr_scheduler.CosineAnnealingLR(ft_optimizer, T_max=7)

print("\nPhase 2: Fine-tuning (unfreeze last 2 MBConv blocks, 7 epochs)...")
t0 = time.time()
for epoch in range(1, 8):
    tl, _ = train_loop(tl_model, tl_train, tl_criterion, ft_optimizer, epoch)
    vl, _ = eval_loop(tl_model,  tl_val,   tl_criterion)
    tl_train_losses.append(tl); tl_val_losses.append(vl)
    ft_scheduler.step()
print(f"⏱️  Phase 2 done in {time.time()-t0:.1f}s")
plot_history(tl_train_losses, tl_val_losses, 'Transfer Learning — Loss (Phase 1 + 2)')


In [ ]:
# ── 10e. Evaluation & Sample Predictions ─────────────────────────────────────
tl_model.eval()
all_preds, all_labels = [], []
with torch.no_grad():
    for X, y in tl_val:
        adjusted_y = torch.tensor([CLASS_MAP.get(yi.item(), yi.item()) for yi in y])
        preds = tl_model(X.to(DEVICE)).argmax(1).cpu()
        all_preds.extend(preds.numpy()); all_labels.extend(adjusted_y.numpy())

cm = confusion_matrix(all_labels, all_preds)
fig, ax = plt.subplots(figsize=(6, 5))
ConfusionMatrixDisplay(cm, display_labels=CLASS_NAMES).plot(ax=ax, colorbar=False)
ax.set_title('Transfer Learning — CIFAR-10 Confusion Matrix')
plt.tight_layout(); plt.show()
free_gpu()


---
## 11. Key Concepts Cheat Sheet <a id='11-cheat-sheet'></a>

### Architecture Selection Guide

| Task | Recommended Architecture | Why |
|------|--------------------------|-----|
| Tabular / structured data | MLP | Flexible, no spatial assumption |
| Image classification | CNN | Exploits spatial locality |
| Sequence classification | LSTM / GRU / Transformer | Captures temporal dependencies |
| Time-series regression | LSTM / GRU | State memory across timesteps |
| Anomaly detection | Autoencoder | High recon error = anomaly |
| Controllable generation | VAE | Smooth latent space |
| Photorealistic generation | GAN | Adversarial sharpness |
| Small labelled dataset | Transfer Learning | Leverage pre-trained features |

---
### Common Activation Functions

| Function | Formula | Use case |
|----------|---------|----------|
| **ReLU** | max(0, x) | Default for hidden layers |
| **LeakyReLU** | max(αx, x) | GAN discriminators |
| **Sigmoid** | 1/(1+e^-x) | Binary output |
| **Tanh** | (e^x - e^-x)/(e^x + e^-x) | GAN generator output |
| **Softmax** | e^xi / Σe^xj | Multi-class output |

---
### Optimiser Cheat Sheet

| Optimiser | Best For | Notes |
|-----------|----------|-------|
| **SGD + momentum** | CNNs on large datasets | Often best final accuracy |
| **Adam** | Most tasks | Fast convergence, good default |
| **AdamW** | Transformers | Adam + weight decay |
| **RMSProp** | RNNs | Adapts LR per parameter |

---
### GTX 1650 4 GB VRAM — Memory Budget

| Model type | Typical VRAM (batch=64) |
|------------|------------------------|
| MLP (MNIST) | ~100 MB |
| LeNet CNN | ~200 MB |
| LSTM (seq) | ~150 MB |
| VAE | ~120 MB |
| Mini DCGAN | ~600 MB |
| MobileNetV2 (fine-tune) | ~1.5 GB |

> Use `torch.cuda.memory_allocated()` and `torch.cuda.memory_reserved()` to monitor.

---
### Regularisation Toolkit

| Technique | Prevents | Usage |
|-----------|----------|-------|
| **Dropout** | Overfitting | p=0.2–0.5 on hidden layers |
| **Batch Norm** | Internal covariate shift | After Conv / Linear, before activation |
| **Weight Decay (L2)** | Large weights | optimizer weight_decay=1e-4 |
| **Data Augmentation** | Overfitting | `torchvision.transforms` |
| **Early Stopping** | Over-training | Stop when val loss stops improving |

---
*Notebook authored for corporate deep learning training. All models train on GTX 1650 (4 GB VRAM) in under 5 minutes total.*


In [ ]:
# ── Final Memory Summary ──────────────────────────────────────────────────────
if DEVICE.type == 'cuda':
    print(f"Peak VRAM allocated: {torch.cuda.max_memory_allocated()/1e6:.1f} MB")
    print(f"Peak VRAM reserved : {torch.cuda.max_memory_reserved()/1e6:.1f} MB")
else:
    print("Running on CPU — no VRAM to report.")

print("\n🎉 Notebook complete! All models trained successfully.")
print("   Sections covered:")
for i, s in enumerate([
    "Environment Setup", "MLP", "CNN", "Vanilla RNN", "LSTM",
    "GRU", "Autoencoder", "VAE", "GAN", "Transfer Learning"], 1):
    print(f"   {i:2d}. {s}")
